# Relatório de Faltantes

Espécies que **não foram encontradas** em alguma das três APIs (Flora, IUCN, GBIF).

Cada linha representa um par `(ESPÉCIE, API)` que falhou. Motivos possíveis:
- `not found` — a API retornou sem resultados para o nome científico
- `no match` — GBIF não conseguiu resolver o nome para um táxon
- `HTTPError: ...` — falha de rede ou resposta inválida

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path().resolve().parent
MISSING = ROOT / 'data' / 'C1_faltantes.csv'

df = pd.read_csv(MISSING)
print(f'{len(df)} registros de faltantes')
df.head(10)

191 registros de faltantes


,ESPÉCIE,PONTO,FAMÍLIA,REINO,API,MOTIVO
0,Geophagus sp.,LB10,Cichlidae,Animalia,iucn,not found
1,Pirinampus pirinampu,LB10,Pimelodidae,Animalia,iucn,not found
2,Poecilia sp.1,LB2,Poeciliidae,Animalia,iucn,not found
3,Poecilia sp.2,LB2,Poeciliidae,Animalia,iucn,not found
4,Poecilia sp.3,LB2,Poeciliidae,Animalia,iucn,not found
5,Pristella maxilaris,LB4,Acestrorhamphidae,Animalia,iucn,not found
6,Saxatilia saxatilis,LB4,Cichlidae,Animalia,gbif,no match
7,Saxatilia saxatilis,LB4,Cichlidae,Animalia,iucn,not found
8,Poecilia sp.,LB4,Poeciliidae,Animalia,iucn,not found
9,Hyphessobrycon sp.,LB4,Acestrorhamphidae,Animalia,iucn,not found


## Resumo por API

In [2]:
df.groupby(['API', 'MOTIVO']).size().to_frame('ocorrências').sort_values('ocorrências', ascending=False)

,,ocorrências
API,MOTIVO,
iucn,not found,183
gbif,no match,8


## Espécies únicas ausentes em cada API

In [3]:
for api in sorted(df['API'].unique()):
    sub = df[df['API'] == api]
    print(f'\n=== {api.upper()} — {sub["ESPÉCIE"].nunique()} espécies ===')
    display(sub[['ESPÉCIE', 'FAMÍLIA', 'REINO', 'MOTIVO']].drop_duplicates().head(50))


=== GBIF — 8 espécies ===


,ESPÉCIE,FAMÍLIA,REINO,MOTIVO
6,Saxatilia saxatilis,Cichlidae,Animalia,no match
19,Saxatilia lepidota,Cichlidae,Animalia,no match
26,Saxatilia sp.,Cichlidae,Animalia,no match
86,Macronema sp.,Hydropsychidae,Animalia,no match
91,Leptonema sp.,Hydropsychidae,Animalia,no match
99,Orthocladiinae sp,Chironomidae,Animalia,no match
102,Chironominae sp,Chironomidae,Animalia,no match
189,Heliconia,Heliconiaceae,NaN,no match



=== IUCN — 183 espécies ===


,ESPÉCIE,FAMÍLIA,REINO,MOTIVO
0,Geophagus sp.,Cichlidae,Animalia,not found
1,Pirinampus pirinampu,Pimelodidae,Animalia,not found
2,Poecilia sp.1,Poeciliidae,Animalia,not found
3,Poecilia sp.2,Poeciliidae,Animalia,not found
4,Poecilia sp.3,Poeciliidae,Animalia,not found
5,Pristella maxilaris,Acestrorhamphidae,Animalia,not found
7,Saxatilia saxatilis,Cichlidae,Animalia,not found
8,Poecilia sp.,Poeciliidae,Animalia,not found
9,Hyphessobrycon sp.,Acestrorhamphidae,Animalia,not found
10,Hemigrammus cf. vonderwinkleri,Acestrorhamphidae,Animalia,not found


## Padrão comum: nomes com `sp.` ou `cf.`

Nomes não-binomiais (ex: `Poecilia sp.`, `Sympetrum cf.`) não têm avaliação IUCN/Flora porque representam morfoespécies ou identificações incompletas.

In [4]:
padroes = df[df['ESPÉCIE'].str.contains(r'\bsp\.?\d*\b|\bcf\.?\b', regex=True, na=False)]
print(f'{padroes["ESPÉCIE"].nunique()} nomes não-binomiais de {df["ESPÉCIE"].nunique()} únicos')
padroes[['ESPÉCIE', 'API', 'MOTIVO']].drop_duplicates().head(30)

90 nomes não-binomiais de 184 únicos


,ESPÉCIE,API,MOTIVO
0,Geophagus sp.,iucn,not found
2,Poecilia sp.1,iucn,not found
3,Poecilia sp.2,iucn,not found
4,Poecilia sp.3,iucn,not found
8,Poecilia sp.,iucn,not found
9,Hyphessobrycon sp.,iucn,not found
10,Hemigrammus cf. vonderwinkleri,iucn,not found
12,Microcharacidium sp.,iucn,not found
13,Hemigrammus sp.,iucn,not found
15,Bryconops sp.,iucn,not found


## Faltantes "reais" (nomes binomiais completos)

Essas são as que merecem revisão — nome parece válido mas nenhuma API encontrou.

In [5]:
mask_nao_binomial = df['ESPÉCIE'].str.contains(r'\bsp\.?\d*\b|\bcf\.?\b', regex=True, na=False)
reais = df[~mask_nao_binomial]
reais[['ESPÉCIE', 'FAMÍLIA', 'REINO', 'API', 'MOTIVO']].drop_duplicates().sort_values(['API', 'ESPÉCIE'])

,ESPÉCIE,FAMÍLIA,REINO,API,MOTIVO
189,Heliconia,Heliconiaceae,NaN,gbif,no match
19,Saxatilia lepidota,Cichlidae,Animalia,gbif,no match
6,Saxatilia saxatilis,Cichlidae,Animalia,gbif,no match
49,Acanthalagma luteum,Coenagrionidae,Animalia,iucn,not found
18,Acestrorhynchus gen1,Acestrorhynchidae,Animalia,iucn,not found
...,...,...,...,...,...
36,Telebasis demarara,Coenagrionidae,Animalia,iucn,not found
46,Tramea darwini,Libellulidae,Animalia,iucn,not found
153,Urospatha sagittifolia,Araceae,Plantae,iucn,not found
173,Varronia curassavica,Cordiaceae,Plantae,iucn,not found
